# Plant Detection using TensorFlow Object Detection API

This notebook fine-tunes an object detection model for plant instance
detection on the PhenoBench dataset.

- Model: SSD MobileNet V2 (320×320)
- Framework: TensorFlow Object Detection API (TF 2.11)
- Task: Crop vs Weed detection

All training, curve, and export plumbing is delegated to the shared
`agri_vision_edge.tfod_trainer` package, so this notebook only carries the
experiment-specific bits: the rich `FineTuneConfig`, the `ExperimentManifest`,
publication-quality curves, and qualitative evaluation. The same run can be
driven head-less from Python (see `notebooks/finetuning.py` for an interactive
marimo front-end over the identical `FinetuneRunConfig` + `run_finetune`).

The notebook covers:
1. Environment setup
2. Dataset loading
3. Model configuration
4. Training and evaluation
5. Qualitative inference results

## Environment Setup

This experiment is designed for:
- Python 3.10
- TensorFlow 2.11
- CUDA-enabled GPU

On Kaggle:
Enable "Pin to original environment" to avoid version drift.

In [ ]:
!python -V

In [ ]:
!pip install --no-cache-dir --no-deps \
  tf_slim \
  pycocotools \
  lvis \
  contextlib2 \
  gin-config \
  tf-models-official==2.13.2 \
  git+https://github.com/frdiener/agri-vision-edge.git

In [ ]:
import json
from pathlib import Path
from dataclasses import asdict
import matplotlib.pyplot as plt

# setup_tensorflow_models() must run before anything pulls in object_detection.
from agri_vision_edge.third_party import setup_tensorflow_models
setup_tensorflow_models()

from agri_vision_edge.experiment import (
    ExperimentManifest,
    capture_environment,
)
from agri_vision_edge.experiment import (
    AugmentationConfig,
    FineTuneConfig,
)

# The shared trainer: one config object drives finetune / QAT + export.
from agri_vision_edge.tfod_trainer import (
    FinetuneRunConfig,
    run_finetune,
    write_pipeline,
    export_run,
)

# Curves are read back from the trainer's metrics_history.json (no TensorBoard).
from agri_vision_edge.evaluation.curves import (
    load_history_scalars,
    available_tags,
    plot_metric_curves,
    plot_loss_curves,
    plot_learning_rate,
    plot_steps_per_second,
    plot_map_curves,
    plot_recall_curves,
)

In [ ]:
EXPERIMENT_ROOT = Path("/kaggle/working")

# tfod_trainer writes the as-run pipeline to <output_dir>/finetune/pipeline.config,
# checkpoints + metrics_history.json to <output_dir>/train, and the export to
# <output_dir>/export. We point output_dir at the experiment root.
OUTPUT_DIR = EXPERIMENT_ROOT

GRAPHS_PATH = EXPERIMENT_ROOT / "graphs"
MANIFEST_PATH = EXPERIMENT_ROOT / "manifest.json"

GRAPHS_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
manifest = ExperimentManifest(
    name="ssd-mobilenet-v2_sc_phenobench_320x320",
    task="object_detection",
)

manifest.set_environment(
    platform="kaggle",
    **capture_environment(),
)

In [ ]:
config = FineTuneConfig(
    batch_size=16,
    learning_rate_base=0.001,
    warmup_learning_rate=0.0003,

    num_steps=30_000,

    warmup_steps=1000,

    early_stopping_patience=60,
    early_stopping_min_delta=0.0,

    image_size=320,

    #
    # Anchor tuning
    #

    anchor_min_scale=0.02,
    anchor_max_scale=0.28,

    anchor_aspect_ratios=(
        0.33,
        0.5,
        1.0,
        2.0,
        3.0,
    ),

    #
    # Matcher thresholds
    #

    matched_threshold=0.4,
    unmatched_threshold=0.3,

    #
    # Data augmentation
    #

    augmentation=AugmentationConfig(
        #
        # Crop
        #

        random_crop=True,

        crop_min_object_covered=1.0,
        crop_min_area=0.6,
        crop_max_area=1.0,
        crop_overlap_thresh=0.3,

        #
        # Geometric invariance
        #

        horizontal_flip=True,
        horizontal_flip_probability=0.5,

        vertical_flip=True,
        vertical_flip_probability=0.5,

        rotation90=True,
        rotation90_probability=0.5,

        #
        # Scale / zoom invariance
        #

        zoom_range=(
            0.8,
            1.2,
        ),

        #
        # Photometric augmentation
        #

        brightness_max_delta=0.15,

        contrast_range=(
            0.8,
            1.2,
        ),

        saturation_range=(
            0.8,
            1.2,
        ),

        hue_max_delta=0.2,

        #
        # Compression robustness
        #

        # jpeg_quality_range=(50,100),
    ),

    #
    # NMS
    #

    nms_score_threshold=0.05,

    nms_iou_threshold=0.5,

    max_detections_per_class=100,
    max_total_detections=100,
)

manifest.add_stage(
    "finetune",
    config=asdict(config),
)

'defined'

## Dataset

We use preprocessed TFRecord files derived from PhenoBench.

Additionally, raw images are used for qualitative evaluation.

In [ ]:
manifest.set_dataset(
    name="phenobench",

    train_split="train",

    validation_split="val",

    num_classes=1,
)

# A dataset bundle for tfod_trainer is a directory holding label_map.pbtxt,
# train.record and val.record -- which is exactly the layout below.
dataset_dir = Path("/kaggle/input/datasets/freimutdiener/sc-phenobench")
dataset_raw_dir = Path("/kaggle/input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench")

label_map_path = dataset_dir / "label_map.pbtxt"

test_imgs = list((dataset_raw_dir / "test" / "images").glob("*.png"))

print(f"{len(test_imgs)} test images loaded")

## Model Configuration

We fine-tune a pre-trained SSD model:
- Backbone: MobileNetV2
- Input size: 320×320

We adapt:
- number of classes
- learning rate schedule
- dataset paths

Rather than configure the TFOD pipeline by hand, we hand the base model, the
dataset bundle and the `FineTuneConfig` above to a single `FinetuneRunConfig`;
the trainer renders the as-run `pipeline.config` from them.

In [ ]:
model_name = "ssd_mobilenet_v2_320x320_coco17_tpu-8"

manifest.set_checkpoint(
    model_name=model_name,
    pretrained_dataset="coco17",

    source="tensorflow_model_zoo",
)

# Base model in the TF model-zoo layout (pipeline.config + checkpoint/ckpt-0.*).
MODEL_DIR = (Path("/kaggle/input/models/freimutdiener/"
                  "ssd-mobilenet-v2-320x320/tensorflow2/coco17/1")
             / model_name)

In [ ]:
run_config = FinetuneRunConfig(
    model_path=MODEL_DIR,
    dataset_bundle_path=dataset_dir,
    num_classes=1,
    output_dir=OUTPUT_DIR,
    finetune=config,
    # qat_scheme="full",  # set to run QAT (fold_bn / reset_optimizer auto-on)
)

# Render the as-run pipeline now so it can be inspected (and registered as an
# artifact) before committing to training. run_finetune() would write it anyway.
write_pipeline(run_config)

manifest.add_artifact(
    "finetune/pipeline.config",

    artifact_type="pipeline_config",

    stage="finetune",
)

print("Pipeline:", run_config.pipeline_config_path)

## Training

`run_finetune` builds the detection model, restores the COCO checkpoint,
applies any graph modifications (BN folding / QAT when `qat_scheme` is set),
and runs the training loop with metric-based checkpointing and early stopping.
It returns handles to the produced artifacts.

In [ ]:
result = run_finetune(run_config)

In [ ]:
best = json.loads(result.best_metric_path.read_text())

print(
    f"Best {best['metric_name']}: {best['metric_value']:.5f} "
    f"at step {best['step']}"
)

manifest.update_stage(
    "finetune",

    metrics={"best_metric": best},
)

manifest.update_stage(
    "finetune",

    artifacts={"best_metric_json": "train/best_metric.json"},
)

## Training Metrics and Learning Curves

The trainer appends one flat record per logged step to `metrics_history.json`
(no TensorBoard event files). `load_history_scalars` turns that into the same
tidy long-format frame the `plot_*` helpers consume, so the curves below render
straight from the trainer's history.

These figures are exported as PDF for inclusion in reports, presentations, or
academic theses.

In [ ]:
history_df = load_history_scalars(result.history_path)

print("Available metric tags:")
for tag in available_tags(history_df):
    print("-", tag)

In [ ]:
loss_fig, _ = plot_loss_curves(
    history_df,
    smoothing=0.6,
    save_path=GRAPHS_PATH / "training_loss_curves.pdf",
)

display(loss_fig)

In [ ]:
lr_fig, _ = plot_learning_rate(
    history_df,
    smoothing=0.6,
    save_path=GRAPHS_PATH / "learning_rate_schedule.pdf",
)

display(lr_fig)

In [ ]:
tput_fig, _ = plot_steps_per_second(
    history_df,
    smoothing=0.5,
    save_path=GRAPHS_PATH / "training_throughput.pdf",
)

display(tput_fig)

The generated figures are also exported as PDF files:

- `training_loss_curves.pdf`
- `learning_rate_schedule.pdf`
- `training_throughput.pdf`

These vector graphics are suitable for direct inclusion in scientific
publications and LaTeX-based theses.

## Validation Metrics

The same history holds the COCO-style validation metrics evaluated at each
logged step, so precision / recall curves come from `history_df` too -- no
separate event log to parse.

In [ ]:
map_fig, _ = plot_map_curves(
    history_df,
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_precision_curves.pdf",
)

display(map_fig)

In [ ]:
recall_fig, _ = plot_recall_curves(
    history_df,
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_recall_curves.pdf",
)

display(recall_fig)

In [ ]:
size_precision_fig, _ = plot_metric_curves(
    df=history_df,
    tags=[
        "DetectionBoxes_Precision/mAP (small)",
        "DetectionBoxes_Precision/mAP (medium)",
        "DetectionBoxes_Precision/mAP (large)",
    ],
    title="Validation Precision per Size",
    ylabel="mAP",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_precision_size_curves.pdf",
)

display(size_precision_fig)

In [ ]:
size_recall_fig, _ = plot_metric_curves(
    df=history_df,
    tags=[
        "DetectionBoxes_Recall/AR@100 (small)",
        "DetectionBoxes_Recall/AR@100 (medium)",
        "DetectionBoxes_Recall/AR@100 (large)",
    ],
    title="Validation Recall per Size",
    ylabel="Average Recall",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_recall_size_curves.pdf",
)

display(size_recall_fig)

In [ ]:
manifest.add_artifact(
    "graphs",

    artifact_type="evaluation_plots",

    stage="finetune",
)

## Model Export

`export_run` exports the best checkpoint to the standard TF model-zoo layout:

```
export/
├── checkpoint/ckpt-0.*   # model-only, restorable as a "detection" checkpoint
├── pipeline.config
└── saved_model/          # fp32 SavedModel for test inference
```

The fp32 SavedModel is used for the qualitative evaluation below. The exported
checkpoint is drop-in usable as the `model_path` of a follow-up
`FinetuneRunConfig`, which makes *resuming QAT from this finetune* identical to
finetuning from the COCO17 checkpoint.

INT8 TFLite conversion is a separate step handled by
`notebooks/tflite_conversion.py`, which consumes this export.

In [ ]:
export_result = export_run(run_config)

print("Export dir: ", export_result.export_dir)
print("SavedModel: ", export_result.saved_model_dir)
print("Checkpoint: ", export_result.checkpoint)
print("Pipeline:   ", export_result.pipeline_config)

## Register artifacts and save Manifest

In [ ]:
manifest.add_artifact(
    "train",
    artifact_type="tfod_train_dir",
    stage="finetune",
)

manifest.add_artifact(
    "train/metrics_history.json",
    artifact_type="metrics_history",
    stage="finetune",
)

manifest.add_artifact(
    "export/saved_model",
    artifact_type="tfod_saved_model",
    stage="finetune",
)

manifest.add_artifact(
    "export/checkpoint",
    artifact_type="tfod_checkpoint",
    stage="finetune",
)

manifest.save(MANIFEST_PATH)

## Qualitative Evaluation

We visualize predictions on unseen test images using the exported fp32
SavedModel.

In [ ]:
%matplotlib inline

from agri_vision_edge.tfod.inference import (
    load_saved_model,
    load_label_map,
    detect_image,
)

detect_fn = load_saved_model(str(export_result.saved_model_dir))

category_index = load_label_map(label_map_path)

for image_path in test_imgs[:10]:

    vis, _ = detect_image(
        detect_fn=detect_fn,
        image_path=image_path,
        category_index=category_index,
        image_size=config.image_size,
        score_threshold=0.5,
        max_boxes=60,
    )

    plt.figure(figsize=(16, 16))

    plt.imshow(vis)

    plt.axis("off")

    plt.show()

    plt.close()

## Discussion

The model demonstrates:

- Successful localization of plant instances
- Overlapping detections reduced via NMS
- Sensitivity to small objects (PhenoBench-specific challenge)

Limitations:
- Performance depends strongly on resolution (320×320)
- Dense scenes produce multiple candidate detections
- Further improvements possible via:
  - anchor tuning
  - longer training
  - quantization-aware training (QAT) -- set `qat_scheme` on the
    `FinetuneRunConfig`, or resume QAT from this finetune's export.

Future work:
- INT8 deployment via TFLite (`notebooks/tflite_conversion.py`)
- Real-time inference optimization